# Product search and similar products

A catalog of 20,000 Amazon products, 150 real customer queries, and for each one human relevance judgments: E (*Exact*, the product asked for), S (*Substitute*, an acceptable substitute), C (*Complement*, an accessory, such as a case for a phone) or I (*Irrelevant*).

The dataset is made of three Parquet files, loaded below into `products`, `queries` and `qrels`:

- `products.parquet`: the catalog, one product per row. `product_id` (the Amazon ID), `title`, `brand`, `color`, `bullets` (the listing's bullet points) and `description` (stripped of HTML and cut at 600 characters). Some fields are empty: the color and the description often are. The lab only uses the title, the brand and the bullet points
- `queries.parquet`: the 150 queries, one per row. `query_id` and `query` (the text the customer typed, typos included)
- `qrels.parquet`: the relevance judgments, one row per judged (query, product) pair. `query_id`, `product_id` and `esci_label` (E, S, C or I). Each query has between 8 and 43 judged products, 3,728 judgments in all. A product with no judgment for a query counts as irrelevant to it

1. Write the evaluation measures
2. A baseline: BM25
3. Dense search, exact then approximate (HNSW)
4. Hybrid search
5. Reranking with a cross-encoder
6. Compare, and read the failures
7. Similar products

This notebook runs faster with a GPU: *Runtime > Change runtime type > T4 GPU*. Without one, it still runs: the two computations too long for a CPU, encoding the catalog (exercise 3) and reranking 100 queries (exercise 5), run on a sample there, then their result computed on a GPU is loaded instead.

## Data and models

In [ ]:
!wget -q https://github.com/shuuchuu/datasets/raw/refs/heads/main/ghsparse.sh -O /tmp/ghsparse.sh && bash /tmp/ghsparse.sh esci-products-subset
%pip install -q bm25s faiss-cpu

In [ ]:
import os
import time

# bm25s uses JAX when it's installed (as on Colab), and JAX grabs 75% of the GPU's memory
# up front: keep it on the CPU, the GPU is for the models.
os.environ["JAX_PLATFORMS"] = "cpu"

import numpy as np
import pandas as pd
import torch
from sentence_transformers import CrossEncoder, SentenceTransformer

products = pd.read_parquet("esci-products-subset/products.parquet")
queries = pd.read_parquet("esci-products-subset/queries.parquet")
qrels = pd.read_parquet("esci-products-subset/qrels.parquet")
products["text"] = products.title + ". " + products.brand + ". " + products.bullets
print(len(products), "products,", len(queries), "queries,", len(qrels), "judgments")

GPU = torch.cuda.is_available()
DEVICE = "cuda" if GPU else "cpu"
# On a CPU, float32: half precisions (float16, and bfloat16, the published weights' one,
# which we'd get by default) are very slow there.
DTYPE = torch.float16 if GPU else torch.float32
print("GPU" if GPU else "No GPU: the long computations will load their result")

embedder = SentenceTransformer(
  "Qwen/Qwen3-Embedding-0.6B", device=DEVICE, model_kwargs={"dtype": DTYPE}
)
embedder.max_seq_length = (
  128  # the start of a product listing is enough, and encoding is twice as fast
)
reranker = CrossEncoder(
  "BAAI/bge-reranker-v2-m3",
  device=DEVICE,
  max_length=512,
  model_kwargs={"dtype": DTYPE},
)
print("Models loaded")

## 1. Writing the evaluation measures

For each query, a search engine returns an ordered list of product IDs. We measure it with:

- **recall at 100**: the share of the query's *Exact* products found in the top 100 results (what a reranking step will see)
- **nDCG at 10**: the quality of the top 10 results (what the customer sees). Each result brings a gain according to its judgment (E: 1, S: 0.1, C: 0.01, I or unjudged: 0), divided by $\log_2(\text{rank} + 1)$; the sum (DCG) is divided by that of the best possible ranking (ideal DCG)

Write `recall_at` and `ndcg_at`. `gains[query_id]` maps each product judged for a query to its gain.

**Useful documentation**

- [Discounted cumulative gain](https://en.wikipedia.org/wiki/Discounted_cumulative_gain)
- [`numpy.log2`](https://numpy.org/doc/stable/reference/generated/numpy.log2.html)
- [`sorted`](https://docs.python.org/3/library/functions.html#sorted)

In [ ]:
GAIN = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}
gains = {
  query_id: dict(zip(group.product_id, group.esci_label.map(GAIN)))
  for query_id, group in qrels.groupby("query_id")
}


def recall_at(ranking: list[str], query_gains: dict[str, float], k: int = 100) -> float:
  # Your code here
  ...


def ndcg_at(ranking: list[str], query_gains: dict[str, float], k: int = 10) -> float:
  # Your code here
  ...

In [ ]:
def evaluate(run: dict[int, list[str]]) -> pd.Series:
  """The mean recall@100 and nDCG@10 of a run: query ID -> ranked product IDs."""
  return pd.Series(
    {
      "recall@100": np.mean([recall_at(run[q], gains[q]) for q in run]),
      "nDCG@10": np.mean([ndcg_at(run[q], gains[q]) for q in run]),
    }
  )

### Additional hints

The calls and parameters to use:

- a query's *Exact* products are those whose gain is `1.0`
- `query_gains.get(product, 0.0)`: a product's gain, 0 if it wasn't judged
- `np.log2(rank + 1)`: the discount; ranks start at 1 (`enumerate(ranking[:k], start=1)`)
- the ideal DCG: the same sum over `sorted(query_gains.values(), reverse=True)[:k]`
- to check, with `toy = {"a": 1.0, "b": 0.1, "c": 0.0, "d": 1.0}` ("a" and "d" are *Exact*, "b" is a *Substitute*): `recall_at(["a", "x", "d"], toy)` is `1.0`, `recall_at(["a", "x", "y"], toy, k=2)` is `0.5`, `ndcg_at(["a", "d", "b"], toy)` is `1.0` and `ndcg_at(["x", "a", "d"], toy)` is about `0.673`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
def recall_at(ranking: list[str], query_gains: dict[str, float], k: int = 100) -> float:
  # The query's Exact products: those whose gain is 1.0
  exact = {product for product, gain in query_gains.items() if ...}
  # The ones among the first k results
  found = exact & set(ranking[:...])
  return ...  # the share of the Exact products that were found


def ndcg_at(ranking: list[str], query_gains: dict[str, float], k: int = 10) -> float:
  # DCG: each of the first k results brings its gain (0 if it wasn't judged: query_gains.get(product, 0.0)),
  # divided by log2(rank + 1)
  dcg = sum(... for rank, product in enumerate(ranking[:k], start=1))
  # Ideal DCG: the same sum over the best possible ranking, i.e. the query's gains by decreasing value
  ideal = ...
  idcg = sum(... for rank, gain in enumerate(ideal, start=1))
  return ...
```

### Solution

In [ ]:
def recall_at(ranking: list[str], query_gains: dict[str, float], k: int = 100) -> float:
  exact = {product for product, gain in query_gains.items() if gain == 1.0}
  return len(exact & set(ranking[:k])) / len(exact)


def ndcg_at(ranking: list[str], query_gains: dict[str, float], k: int = 10) -> float:
  dcg = sum(
    query_gains.get(product, 0.0) / np.log2(rank + 1)
    for rank, product in enumerate(ranking[:k], start=1)
  )
  ideal = sorted(query_gains.values(), reverse=True)[:k]
  idcg = sum(gain / np.log2(rank + 1) for rank, gain in enumerate(ideal, start=1))
  return dcg / idcg

## 2. A baseline: BM25

[`bm25s`](https://github.com/xhluca/bm25s) indexes a corpus for BM25 in seconds. The next cell defines `bm25_search(k1, b)`, which indexes the products' texts with these two parameters and returns each query's top 100 products, as `{query_id: [product_id, ...]}`. With the default values of `bm25s`, $k_1 = 1.5$ and $b = 0.75$, it gives `runs["BM25"]`, the baseline of the next exercises.

These two parameters of the formula control the **saturation** ($k_1$: the smaller it is, the less a word repeated in a listing adds) and the **length normalization** ($b$, between 0 and 1: at 0, a listing's length is ignored; at 1, it weighs fully). The catalog's listings range from a few words to over 150. Compare the default values, `b=0` (no length normalization), `k1=0.5` and `k1=3` on nDCG@10. What does each parameter control, and which one matters most here?

In [ ]:
import bm25s

runs = {}

# The texts, tokenized once and for all: only the index depends on k1 and b.
product_tokens = bm25s.tokenize(
  products.text.tolist(), stopwords="en", show_progress=False
)
query_tokens = bm25s.tokenize(
  queries["query"].tolist(), stopwords="en", show_progress=False
)


def bm25_search(k1: float = 1.5, b: float = 0.75) -> dict[int, list[str]]:
  """Each query's top 100 products according to BM25: query ID -> product IDs."""
  retriever = bm25s.BM25(k1=k1, b=b)
  retriever.index(product_tokens, show_progress=False)
  # `indices`: the positions of the 100 best products in the catalog, one row per query
  indices, _ = retriever.retrieve(query_tokens, k=100, show_progress=False)
  return {
    query_id: products.product_id.iloc[row].tolist()
    for query_id, row in zip(queries.query_id, indices)
  }


runs["BM25"] = bm25_search()  # the default values of bm25s: k1 = 1.5 and b = 0.75
print(evaluate(runs["BM25"]))

In [ ]:
# Your code here

### Additional hints

The calls and parameters to use:

- `bm25_search(k1=..., b=...)` returns a ranking, `{query_id: [product_id, ...]}`, which `evaluate` measures; `evaluate(run)["nDCG@10"]` keeps its nDCG@10
- one parameter changes at a time, the other keeps its default value: `(k1, b)` is `(1.5, 0.75)`, `(1.5, 0)`, `(0.5, 0.75)` then `(3, 0.75)`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
# One parameter changes at a time: the default values, b=0, k1=0.5 then k1=3
for k1, b in [(1.5, 0.75), (1.5, 0), (0.5, 0.75), (3, 0.75)]:
  ndcg = ...  # the nDCG@10 of BM25's ranking with these two parameters
  print(f"k1={k1}, b={b}: nDCG@10 {ndcg:.3f}")
```

### Solution

In [ ]:
for k1, b in [(1.5, 0.75), (1.5, 0), (0.5, 0.75), (3, 0.75)]:
  ndcg = evaluate(bm25_search(k1=k1, b=b))["nDCG@10"]
  print(f"k1={k1}, b={b}: nDCG@10 {ndcg:.3f}")

# k1 controls saturation: the smaller it is, the sooner a repeated word stops adding.
# b controls length normalization: at 0, a long listing, which holds more words, gets
# ahead of an equally relevant short one. Here b matters most: without length
# normalization, nDCG@10 drops from 0.473 to 0.430, while k1 only moves it from 0.455
# (k1=3) to 0.480 (k1=0.5). Fast saturation helps a little: a query word coming back
# often in a listing says little more about it.

## 3. Dense search

Encode the products (`embedder.encode_document`) and the queries (`embedder.encode_query`). The vectors are normalized: the cosine is a dot product. Find each query's 100 closest products with a matrix product and `np.argsort`, store the result in `runs["dense"]` and evaluate. Without a GPU, `product_texts` and `query_texts` hold 32 products and 16 queries only: the next exercise loads the vectors of all of them, computed on a GPU, and redoes this ranking.

**Useful documentation**

- [`numpy.matmul`](https://numpy.org/doc/stable/reference/generated/numpy.matmul.html)
- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
# Every product and query on a GPU; without one, 32 products and 16 queries (about a
# minute).
product_texts = products.text.tolist() if GPU else products.text[:32].tolist()
query_texts = queries["query"].tolist() if GPU else queries["query"][:16].tolist()

# Your code here

### Additional hints

The calls and parameters to use:

- `embedder.encode_document(texts, batch_size=32, show_progress_bar=True)` for the products (`product_texts`), `embedder.encode_query(texts)` for the queries (`query_texts`)
- `.astype(np.float32)`: on a GPU, the model is in half precision; we compute in `float32`
- `query_vectors @ product_vectors.T`: the scores of every query against every product at once, one row per query
- `np.argsort(-scores, axis=1)[:, :100]`: the positions of each query's 100 best products (we sort the opposite scores to get the best first)
- `products.product_id.iloc[row].tolist()`: the product IDs at those positions

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
start = time.time()
# The products' texts (`product_texts`) and the queries' (`query_texts`)
product_vectors = embedder.encode_document(
  ..., batch_size=32, show_progress_bar=True
).astype(np.float32)
query_vectors = embedder.encode_query(...).astype(np.float32)
print(f"{time.time() - start:.0f} s to encode the catalog")

# One row per query, one column per product
scores = ...
# `top`: for each query, the positions of its 100 best products
top = np.argsort(..., axis=1)[:, :100]
runs["dense"] = {
  query_id: ...  # the IDs of the products at the positions `row`, as a list
  for query_id, row in zip(queries.query_id, top)
}
evaluate(runs["dense"])
```

### Solution

In [ ]:
start = time.time()
product_vectors = embedder.encode_document(
  product_texts, batch_size=32, show_progress_bar=True
).astype(np.float32)
query_vectors = embedder.encode_query(query_texts).astype(np.float32)
print(f"{time.time() - start:.0f} s to encode the catalog")

scores = query_vectors @ product_vectors.T
top = np.argsort(-scores, axis=1)[:, :100]
runs["dense"] = {
  query_id: products.product_id.iloc[row].tolist()
  for query_id, row in zip(queries.query_id, top)
}
evaluate(runs["dense"])

### The HNSW index

Comparing a query with every product is exact, but too expensive with millions of products. An **HNSW** index (a neighborhood graph on several levels) finds the nearest neighbors **approximately**, much faster.

Build a `faiss.IndexHNSWFlat` index, add the products' vectors to it, and search the queries' 100 nearest neighbors. Measure the search time against exact search, done with the same tool (`faiss.IndexFlatIP`, which compares each query with every product), and the share of the 100 exact neighbors HNSW finds. Vary `index.hnsw.efSearch`: what does this parameter control?

**Useful documentation**

- [Faiss: getting started](https://github.com/facebookresearch/faiss/wiki/Getting-started)
- [Faiss: the indexes](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes)
- [`time.time`](https://docs.python.org/3/library/time.html#time.time)

In [ ]:
# Without a GPU, the previous exercise encoded part of the catalog only: we load the
# vectors of every product and query, computed on a GPU by the same code (stored in
# float16), and redo the dense ranking.
if not GPU:
  product_vectors = np.load(
    "esci-products-subset/lab-results/product-embeddings.npy"
  ).astype(np.float32)
  query_vectors = np.load(
    "esci-products-subset/lab-results/query-embeddings.npy"
  ).astype(np.float32)
  top = np.argsort(-(query_vectors @ product_vectors.T), axis=1)[:, :100]
  runs["dense"] = {
    query_id: products.product_id.iloc[row].tolist()
    for query_id, row in zip(queries.query_id, top)
  }
  print(evaluate(runs["dense"]))

# Your code here

#### Additional hints

The calls and parameters to use:

- `faiss.IndexHNSWFlat(dimension, 32, faiss.METRIC_INNER_PRODUCT)`: `dimension` is the vectors' size (`product_vectors.shape[1]`), 32 the number of neighbors each node of the graph is linked to, and the measure is the dot product (the vectors are normalized)
- `index.add(vectors)`, then `index.search(query_vectors, 100)`, which returns the scores and the positions of each query's 100 neighbors
- `index.hnsw.efSearch`: try `16`, `64` and `128`
- `faiss.IndexFlatIP(dimension)`: exact search, with the same `add` and `search` as the HNSW index
- `time.time()` before and after to time it
- the share of exact neighbors found: the intersection between HNSW's 100 positions and those of `top` (the exact search), divided by 100, averaged over the queries

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
import faiss

# 32 is the number of neighbors each node of the graph is linked to
index = faiss.IndexHNSWFlat(
  ..., 32, faiss.METRIC_INNER_PRODUCT
)  # the first argument: the vectors' dimension
index.add(...)  # the products' vectors

for ef in (16, 64, 128):
  index.hnsw.efSearch = ef
  start = time.time()
  _, approximate = index.search(
    ..., 100
  )  # `approximate`: the positions of the 100 neighbors of each query
  elapsed = time.time() - start
  # The share of the 100 exact neighbors (`top`, from exact search) that HNSW also finds, on average over the queries
  overlap = ...
  print(
    f"efSearch {ef:3}: {elapsed * 1000:.0f} ms, {overlap:.1%} of the exact neighbors"
  )

# Exact search, with the same tool, to compare
exact = faiss.IndexFlatIP(...)  # the vectors' dimension
exact.add(...)  # the products' vectors
start = time.time()
...  # the search for the queries' 100 nearest neighbors
print(f"exact search: {(time.time() - start) * 1000:.0f} ms")
```

#### Solution

In [ ]:
import faiss

index = faiss.IndexHNSWFlat(product_vectors.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
index.add(product_vectors)
for ef in (16, 64, 128):
  index.hnsw.efSearch = ef
  start = time.time()
  _, approximate = index.search(query_vectors, 100)
  elapsed = time.time() - start
  overlap = np.mean([len(set(a) & set(b)) / 100 for a, b in zip(approximate, top)])
  print(
    f"efSearch {ef:3}: {elapsed * 1000:.0f} ms, {overlap:.1%} of the exact neighbors"
  )

exact = faiss.IndexFlatIP(product_vectors.shape[1])
exact.add(product_vectors)
start = time.time()
exact.search(query_vectors, 100)
print(f"exact search: {(time.time() - start) * 1000:.0f} ms")

# efSearch is the size of the candidate list explored in the graph: larger is more
# accurate and slower. With 20,000 products, exact search is still fast, barely twice as
# slow as HNSW at efSearch 128; HNSW becomes essential with millions of vectors. A matrix
# product followed by np.argsort would have made it look several times slower: sorting
# the 20,000 scores of each query costs more than computing them.

## 4. Hybrid search

Reciprocal Rank Fusion (RRF) combines several rankings without comparing their scores: each product gets $\sum 1 / (60 + \text{rank})$ over the rankings it appears in (ranks from 1). Write `rrf(*rankings)`, fuse BM25 and dense search for each query into `runs["hybrid"]` (keep the top 100), and evaluate.

**Useful documentation**

- [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf)
- [`collections.defaultdict`](https://docs.python.org/3/library/collections.html#collections.defaultdict)
- [`enumerate`](https://docs.python.org/3/library/functions.html#enumerate)
- [`sorted`](https://docs.python.org/3/library/functions.html#sorted)

In [ ]:
def rrf(*rankings: list[str], k: int = 60) -> list[str]:
  # Your code here
  ...

### Additional hints

The calls and parameters to use:

- `rrf(*rankings, k=60)`: each product gets `1 / (k + rank)` for every ranking it appears in; ranks start at 1 (`enumerate(ranking, start=1)`)
- `defaultdict(float)`: a score of 0 for a product not seen yet
- `sorted(scores, key=scores.get, reverse=True)`: the products by decreasing score
- `rrf(runs["BM25"][query_id], runs["dense"][query_id])[:100]` for each query
- to check: `rrf(["a", "b", "c"], ["b", "c", "d"])` returns `["b", "c", "a", "d"]`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
from collections import defaultdict


def rrf(*rankings: list[str], k: int = 60) -> list[str]:
  scores = defaultdict(float)
  for ranking in rankings:
    for rank, product in enumerate(ranking, start=1):
      scores[product] += ...  # this ranking's contribution: 1 / (k + rank)
  return sorted(scores, key=..., reverse=True)  # the products by decreasing score


runs["hybrid"] = {
  query_id: rrf(..., ...)[:100]  # the BM25 and dense rankings of this query
  for query_id in queries.query_id
}
evaluate(runs["hybrid"])
```

### Solution

In [ ]:
from collections import defaultdict


def rrf(*rankings: list[str], k: int = 60) -> list[str]:
  scores = defaultdict(float)
  for ranking in rankings:
    for rank, product in enumerate(ranking, start=1):
      scores[product] += 1 / (k + rank)
  return sorted(scores, key=scores.get, reverse=True)


runs["hybrid"] = {
  query_id: rrf(runs["BM25"][query_id], runs["dense"][query_id])[:100]
  for query_id in queries.query_id
}
evaluate(runs["hybrid"])

# Fusion finds more exact products in its top 100 (recall@100): that's what the next step,
# reranking, uses. Its nDCG@10 isn't necessarily better than dense search's when BM25 is
# clearly weaker.

## 5. Reranking with a cross-encoder

A cross-encoder reads the query and the product **together**: more accurate, but one model pass per pair. For the first 100 queries (`subset`; without a GPU, the first one only: the next exercise loads the reranking of the 100, computed on a GPU), rerank the top 50 hybrid results with the cross-encoder `reranker`, then append the next 50 in hybrid order. Store the result in `runs["hybrid + reranking"]` and compare with hybrid on the **same** 100 queries.

**Useful documentation**

- [`CrossEncoder.predict`](https://sbert.net/docs/package_reference/cross_encoder/cross_encoder.html#sentence_transformers.cross_encoder.model.CrossEncoder.predict)
- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)

In [ ]:
text_of = dict(zip(products.product_id, products.text))
query_of = dict(zip(queries.query_id, queries["query"]))
# The first 100 queries; without a GPU, 1 (about two minutes).
subset = queries.query_id.iloc[:100] if GPU else queries.query_id.iloc[:1]

# Your code here

### Additional hints

The calls and parameters to use:

- `reranker.predict([(query, text), ...], batch_size=32)`: one (query text, product text) pair per candidate, from `query_of[query_id]` and `text_of[product_id]`; returns an array of scores
- `np.argsort(-scores)`: the candidates' positions by decreasing score
- `evaluate({q: runs["hybrid"][q] for q in subset})`: hybrid on the same 100 queries

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
start = time.time()
reranked = {}
for query_id in subset:
  candidates = runs["hybrid"][query_id][:50]
  # The cross-encoder scores one (query text, product text) pair per candidate
  scores = reranker.predict([(..., ...) for p in candidates], batch_size=32)
  # The candidates by decreasing score (`np.argsort` returns positions)
  order = [candidates[i] for i in ...]
  reranked[query_id] = ...  # the reranked 50, then the next 50 of the hybrid ranking
runs["hybrid + reranking"] = reranked
print(f"{time.time() - start:.0f} s to rerank {len(subset)} queries")

# Compare with hybrid on the same 100 queries
pd.DataFrame(
  {
    "hybrid": ...,
    "hybrid + reranking": ...,
  }
)
```

### Solution

In [ ]:
start = time.time()
reranked = {}
for query_id in subset:
  candidates = runs["hybrid"][query_id][:50]
  scores = reranker.predict(
    [(query_of[query_id], text_of[p]) for p in candidates], batch_size=32
  )
  order = [candidates[i] for i in np.argsort(-scores)]
  reranked[query_id] = order + runs["hybrid"][query_id][50:]
runs["hybrid + reranking"] = reranked
print(f"{time.time() - start:.0f} s to rerank {len(subset)} queries")

pd.DataFrame(
  {
    "hybrid": evaluate({q: runs["hybrid"][q] for q in subset}),
    "hybrid + reranking": evaluate(reranked),
  }
)

## 6. Comparing, and reading the failures

Gather every method's measures in a table, on the 100 reranked queries, and add to it, for each method, the share of the top 10 results that have no judgment. These products count as irrelevant: what is an nDCG of 0.5 worth, then? Then find the queries where BM25 clearly beats dense search (in nDCG@10), and the reverse: read them. What do they have in common?

**Useful documentation**

- [`pandas.DataFrame`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)
- [`DataFrame.nlargest`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nlargest.html) and [`nsmallest`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nsmallest.html)

In [ ]:
# Without a GPU, the previous exercise reranked 1 query only: we load the reranking of the
# 100, computed on a GPU by the same code (each query's products, in order).
if not GPU:
  reranked = pd.read_parquet("esci-products-subset/lab-results/reranked.parquet")
  runs["hybrid + reranking"] = (
    reranked.groupby("query_id").product_id.agg(list).to_dict()
  )
  subset = queries.query_id.iloc[:100]

# Your code here

### Additional hints

The calls and parameters to use:

- `evaluate` takes `{query_id: ranking}`: `{q: run[q] for q in subset}` restricts a run to the 100 queries
- `pd.DataFrame({name: series, ...})`: one column per method; the methods are the keys of `runs`
- `p not in gains[q]`: product `p` wasn't judged for query `q`; `np.mean` of a list of booleans gives the share of `True`
- `table.loc["unjudged@10"] = [...]` adds a row to the table, one value per method, in the order of `runs`
- `ndcg_at(runs["BM25"][q], gains[q])` for each query, and likewise for `runs["dense"]`
- `.nsmallest(8, "gap")`: the 8 smallest gaps (BM25 wins), `.nlargest(8, "gap")` the 8 largest (dense search wins)

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
# One column per method, each evaluated on the same 100 queries (`subset`): `evaluate` takes {query_id: ranking}
table = pd.DataFrame(
  {name: evaluate({q: ... for q in subset}) for name, run in runs.items()}
)
# The share of the top 10 results with no judgment, on average over the queries: one value per method
table.loc["unjudged@10"] = [
  np.mean([np.mean([p not in gains[q] for p in ...]) for q in subset])
  for run in runs.values()
]
print(table.round(3))

# nDCG@10 of BM25 and of dense search, query by query
per_query = pd.DataFrame(
  {
    "query": [query_of[q] for q in queries.query_id],
    "BM25": [ndcg_at(..., gains[q]) for q in queries.query_id],
    "dense": [ndcg_at(..., gains[q]) for q in queries.query_id],
  }
)
per_query["gap"] = ...  # positive when dense search wins, negative when BM25 does
print("BM25 wins:", per_query.nsmallest(8, "gap")["query"].tolist())
print("Dense wins:", ...)  # the 8 queries with the largest gap
```

### Solution

In [ ]:
table = pd.DataFrame(
  {name: evaluate({q: run[q] for q in subset}) for name, run in runs.items()}
)
table.loc["unjudged@10"] = [
  np.mean([np.mean([p not in gains[q] for p in run[q][:10]]) for q in subset])
  for run in runs.values()
]
print(table.round(3))

per_query = pd.DataFrame(
  {
    "query": [query_of[q] for q in queries.query_id],
    "BM25": [ndcg_at(runs["BM25"][q], gains[q]) for q in queries.query_id],
    "dense": [ndcg_at(runs["dense"][q], gains[q]) for q in queries.query_id],
  }
)
per_query["gap"] = per_query.dense - per_query.BM25
print("BM25 wins:", per_query.nsmallest(8, "gap")["query"].tolist())
print("Dense wins:", per_query.nlargest(8, "gap")["query"].tolist())

# About a third of the top 10 results have no judgment, from 29% after reranking to 39%
# for BM25. They count as 0: the DCG is a lower bound, the nDCG most often an
# underestimate (not always, since judging a product also raises the ideal DCG). An
# nDCG of 0.5 may hide relevant products no one judged, and two methods compare badly
# when their shares of unjudged results differ.
# BM25 wins when the query's words are the listing's, proper names included; dense
# search on typos, other languages and rewordings. There are exceptions: dense search
# also wins on a brand (mcm shades men) and on a product name written as one word
# (happypills, "Happy Pills" in the catalog).

## 7. Similar products

On a product's page, we suggest similar products: its nearest neighbors in the embedding space, with no query. Write `similar(product_id, k=5, same_brand=False, min_score=-np.inf, max_score=np.inf)`, which prints the scores and titles of a product's `k` neighbors (without the product itself), optionally restricted to the same brand, whose score lies between `min_score` and `max_score`. Try it on three products of `candidates` (products whose brand has other products in the catalog), with and without `same_brand`, and read the neighbors: what would you suggest to the customer for these three products? Then choose, from the scores you read, a minimum score, below which a neighbor has little to do with the product, and a maximum score, above which it is a variant of the product itself (another size, another color), and run it again.

And since the model is multilingual: search the English catalog with queries in another language, such as French (« casque de moto avec écouteurs », « gamelle pour chien surélevée »).

**Useful documentation**

- [`numpy.argsort`](https://numpy.org/doc/stable/reference/generated/numpy.argsort.html)
- [`numpy.inf`](https://numpy.org/doc/stable/reference/constants.html#numpy.inf)

In [ ]:
position = {
  product: i for i, product in enumerate(products.product_id)
}  # product ID -> row in the catalog

# Products whose brand has other products in the catalog.
brands = products.brand[products.brand != ""].value_counts()
candidates = products[products.brand.isin(brands[brands >= 5].index)]


def similar(
  product_id: str,
  k: int = 5,
  same_brand: bool = False,
  min_score: float = -np.inf,
  max_score: float = np.inf,
) -> None:
  # Your code here
  ...

### Additional hints

The calls and parameters to use:

- `product_vectors @ product_vectors[i]`: the similarity of every product with product `i`
- `scores[i] = -np.inf`: a product isn't its own neighbor
- `products.brand.to_numpy() != products.brand[i]` is `True` for products of another brand: give them `-np.inf` too
- `(scores < min_score) | (scores > max_score)` is `True` for the neighbors outside the two thresholds: give them `-np.inf` too
- `np.argsort(-scores)[:k]`: the positions of the `k` best; `np.isfinite(scores[j])` drops the excluded products (fewer than `k` may remain)
- `candidates.product_id.sample(3, random_state=0)`: three products of `candidates`
- thresholds to try: `min_score=0.5` and `max_score=0.98`
- for a query in French: `product_vectors @ embedder.encode_query(query).astype(np.float32)`, then `products.title.iloc[positions]`

A skeleton to copy into the exercise's cell, replacing each `...`:

```python
def similar(
  product_id: str,
  k: int = 5,
  same_brand: bool = False,
  min_score: float = -np.inf,
  max_score: float = np.inf,
) -> None:
  i = position[product_id]
  scores = product_vectors @ product_vectors[i]  # the similarity to every product
  scores[i] = ...  # a product isn't its own neighbor
  if same_brand:
    # `products.brand.to_numpy() != products.brand[i]` is True for the products of another brand
    scores[...] = -np.inf
  # The neighbors too different (below min_score) and the product's variants (above max_score)
  scores[...] = -np.inf
  print("Product:", products.title[i], "| brand:", products.brand[i])
  for j in ...:  # the positions of the k best scores
    if np.isfinite(scores[j]):  # fewer than k products may be left
      print(f"  {scores[j]:.3f} {products.title[j][:100]}")


for product_id in candidates.product_id.sample(3, random_state=0):
  ...  # similar products, those of the same brand, then with the two thresholds

# Queries in French, against the English catalog
for french in ("casque de moto avec écouteurs", "gamelle pour chien surélevée"):
  scores = (
    product_vectors @ ...
  )  # the query's vector: embedder.encode_query(...), as float32
  print(
    french, "->", products.title.iloc[...].str[:80].tolist()
  )  # the 3 best products' titles
```

### Solution

In [ ]:
def similar(
  product_id: str,
  k: int = 5,
  same_brand: bool = False,
  min_score: float = -np.inf,
  max_score: float = np.inf,
) -> None:
  i = position[product_id]
  scores = product_vectors @ product_vectors[i]
  scores[i] = -np.inf
  if same_brand:
    scores[products.brand.to_numpy() != products.brand[i]] = -np.inf
  scores[(scores < min_score) | (scores > max_score)] = -np.inf
  print("Product:", products.title[i], "| brand:", products.brand[i])
  for j in np.argsort(-scores)[:k]:
    if np.isfinite(scores[j]):
      print(f"  {scores[j]:.3f} {products.title[j][:100]}")


for product_id in candidates.product_id.sample(3, random_state=0):
  similar(product_id)
  similar(product_id, same_brand=True)
  similar(product_id, same_brand=True, min_score=0.5, max_score=0.98)

# The Ekouaer nightshirt's first neighbor (0.998) is the same one in another size: a
# variant, which the maximum score drops (above 0.98, they are mostly other sizes, colors
# or packs of one product). Restricted to the BestLand brand, the palette has only four
# close neighbors, and the fifth, a hair finishing stick at 0.359, has nothing to do with
# it: the minimum score drops it, even if fewer than k products are left. Same brand or
# not depends on who sells: a brand suggests its own range (the Advanced Nutrients
# fertilizers), a marketplace its competitors' products too (the palettes of other
# brands, a House & Garden fertilizer).

for french in ("casque de moto avec écouteurs", "gamelle pour chien surélevée"):
  scores = product_vectors @ embedder.encode_query(french).astype(np.float32)
  print(french, "->", products.title.iloc[np.argsort(-scores)[:3]].str[:80].tolist())